In [1]:
# ============================================
# CELL 1: INSTALL & IMPORT LIBRARIES
# ============================================

# Uncomment nếu bạn chưa cài đặt các thư viện cần thiết
# !pip install lightgbm pyarrow polars

import gc
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import timedelta
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
# ============================================
# CELL 2: CONFIG
# ============================================

DATA_PATH = Path(r"/content/drive/MyDrive/Personal project/Paper/sales_train_evaluation.csv")
MAX_LAGS = 1912
LAST_D = 1913
HORIZON = 28

VALID_DATE = "2016-04-25"
TEST_DATE = "2016-03-28"

STATE = "CA"
FORECAST_DAYS = 28

LGB_PARAMS = {
    "objective": "tweedie",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "tweedie_variance_power": 1.2,
    "learning_rate": 0.075,
    "num_leaves": 2047,
    "min_data_in_leaf": 4095,
    "feature_fraction": 0.5,
    "subsample": 0.7,
    "subsample_freq": 1,
    "max_bin": 100,
    "verbosity": -1,
    "seed": 42
}

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# =================================================
# CELL 3: LOAD DATA (Đường dẫn chuẩn theo ảnh)
# =================================================

calendar = pd.read_csv(r"/content/drive/MyDrive/Personal project/Paper/calendar.csv")
prices = pd.read_csv(r"/content/drive/MyDrive/Personal project/Paper/sell_prices.csv")
sales = pd.read_csv(r"/content/drive/MyDrive/Personal project/Paper/sales_train_evaluation.csv")

print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)
print("Sales shape:", sales.shape)

Calendar shape: (1969, 14)
Prices shape: (6841121, 4)
Sales shape: (30490, 1947)


In [5]:
# ============================================
# CELL 4: MEMORY REDUCTION
# ============================================

def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                else:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                else:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Memory usage: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    return df

calendar = reduce_mem_usage(calendar)
prices = reduce_mem_usage(prices)
sales = reduce_mem_usage(sales)

Memory usage: 0.21 MB -> 0.12 MB
Memory usage: 208.77 MB -> 130.48 MB
Memory usage: 452.91 MB -> 96.13 MB


In [6]:
# ============================================
# CELL 5: FILTER STORE
# ============================================

stores = {
    "CA": ["CA_1", "CA_2", "CA_3", "CA_4"],
    "TX": ["TX_1", "TX_2", "TX_3"],
    "WI": ["WI_1", "WI_2", "WI_3"]
}

STORE_ID = "CA_1"
sales_store = sales[sales["store_id"] == STORE_ID]
print("Filtered store shape:", sales_store.shape)

Filtered store shape: (3049, 1947)


In [7]:
# ============================================
# CELL 6: WIDE TO LONG
# ============================================

id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

sales_long = sales_store.melt(
    id_vars=id_cols,
    var_name="d",
    value_name="sales"
)

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [8]:
# ============================================
# CELL 7: MERGE DATA
# ============================================

calendar["date"] = pd.to_datetime(calendar["date"])

sales_long = sales_long.merge(calendar, on="d", how="left")
sales_long = sales_long.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

# Dọn dẹp RAM sau khi merge diện rộng
gc.collect()

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN


In [9]:
# ============================================
# CELL 8: DATE FEATURES
# ============================================

sales_long["day"] = sales_long["date"].dt.day
sales_long["week"] = sales_long["date"].dt.isocalendar().week.astype(np.int32)
sales_long["month"] = sales_long["date"].dt.month
sales_long["year"] = sales_long["date"].dt.year
sales_long["weekday"] = sales_long["date"].dt.weekday

sales_long["is_weekend"] = (sales_long["weekday"] >= 5).astype(np.int8)

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,day,week,is_weekend
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,NaN,NaN,NaN,0,0,0,NaN,29,4,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,NaN,NaN,NaN,0,0,0,NaN,29,4,1
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,NaN,NaN,NaN,0,0,0,NaN,29,4,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,NaN,NaN,NaN,0,0,0,NaN,29,4,1
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,NaN,NaN,NaN,0,0,0,NaN,29,4,1


In [10]:
# ============================================
# CELL 9: LAG FEATURES
# ============================================

# Sắp xếp theo id và date để đảm bảo lag shift chính xác theo dòng thời gian
sales_long = sales_long.sort_values(by=["id", "date"]).reset_index(drop=True)

LAGS = [7, 14, 28]
for lag in LAGS:
    sales_long[f"lag_{lag}"] = sales_long.groupby("id")["sales"].shift(lag)

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,snap_CA,snap_TX,snap_WI,sell_price,day,week,is_weekend,lag_7,lag_14,lag_28
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,0,0,0,2.0,29,4,1,NaN,NaN,NaN
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,0,0,0,2.0,30,4,1,NaN,NaN,NaN
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,0,0,0,2.0,31,5,0,NaN,NaN,NaN
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,1,1,0,2.0,1,5,0,NaN,NaN,NaN
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,1,0,1,2.0,2,5,0,NaN,NaN,NaN


In [11]:
# ============================================
# CELL 10: ROLLING FEATURES
# ============================================

WINDOWS = [7, 14, 28]
for window in WINDOWS:
    # shift(28) tránh rò rỉ dữ liệu (data leakage) khi dự báo chu kỳ 28 ngày
    sales_long[f"rmean_{window}"] = (
        sales_long.groupby("id")["sales"]
        .transform(lambda x: x.shift(28).rolling(window).mean())
    )
    sales_long[f"rstd_{window}"] = (
        sales_long.groupby("id")["sales"]
        .transform(lambda x: x.shift(28).rolling(window).std())
    )

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,is_weekend,lag_7,lag_14,lag_28,rmean_7,rstd_7,rmean_14,rstd_14,rmean_28,rstd_28
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# ============================================
# CELL 11: PRICE FEATURES
# ============================================

sales_long["price_change"] = sales_long.groupby("id")["sell_price"].pct_change()
sales_long["price_max"] = sales_long.groupby("id")["sell_price"].transform("max")
sales_long["price_norm"] = sales_long["sell_price"] / sales_long["price_max"]

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,lag_28,rmean_7,rstd_7,rmean_14,rstd_14,rmean_28,rstd_28,price_change,price_max,price_norm
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.240234,0.892578
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.240234,0.892578
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.240234,0.892578
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.240234,0.892578
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.240234,0.892578


In [13]:
# ============================================
# CELL 12: ENCODING
# ============================================

from sklearn.preprocessing import LabelEncoder

cat_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]

for col in cat_cols:
    # Fill NaN with a placeholder string to ensure LabelEncoder can process it
    # and then apply Label Encoding to convert strings to unique integers
    sales_long[col] = sales_long[col].fillna("NoEvent")
    le = LabelEncoder()
    sales_long[col] = le.fit_transform(sales_long[col])

# Additionally, fill NaNs for numerical features that might have been introduced
# by lag, rolling, or price feature engineering. Filling with 0 is a common strategy.
numerical_cols_with_nans = [
    'sell_price', 'price_change', 'price_max', 'price_norm',
    'lag_7', 'lag_14', 'lag_28',
    'rmean_7', 'rstd_7', 'rmean_14', 'rstd_14', 'rmean_28', 'rstd_28'
]

for col in numerical_cols_with_nans:
    if col in sales_long.columns:
        sales_long[col] = sales_long[col].fillna(0) # Filling with 0

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,lag_28,rmean_7,rstd_7,rmean_14,rstd_14,rmean_28,rstd_28,price_change,price_max,price_norm
0,FOODS_1_001_CA_1_evaluation,0,0,0,0,0,d_1,3,2011-01-29,11101,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.240234,0.892578
1,FOODS_1_001_CA_1_evaluation,0,0,0,0,0,d_2,0,2011-01-30,11101,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.240234,0.892578
2,FOODS_1_001_CA_1_evaluation,0,0,0,0,0,d_3,0,2011-01-31,11101,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.240234,0.892578
3,FOODS_1_001_CA_1_evaluation,0,0,0,0,0,d_4,1,2011-02-01,11101,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.240234,0.892578
4,FOODS_1_001_CA_1_evaluation,0,0,0,0,0,d_5,4,2011-02-02,11101,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.240234,0.892578


In [14]:
# ============================================
# CELL 13: TRAIN VALID SPLIT
# ============================================

# Đảm bảo định dạng datetime đồng nhất để lọc dữ liệu chính xác
train_data = sales_long[sales_long["date"] < pd.to_datetime(TEST_DATE)]
valid_data = sales_long[(sales_long["date"] >= pd.to_datetime(TEST_DATE)) & (sales_long["date"] < pd.to_datetime(VALID_DATE))]

FEATURES = [
    col for col in sales_long.columns
    if col not in ["sales", "date", "d", "id"]
]

X_train = train_data[FEATURES]
y_train = train_data["sales"]

X_valid = valid_data[FEATURES]
y_valid = valid_data["sales"]

# Explicitly convert the problematical categorical columns to int32
# This ensures LightGBM receives numerical data as expected.
problematic_cat_cols = [col for col in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"] if col in FEATURES]
for col in problematic_cat_cols:
    X_train[col] = X_train[col].astype(np.int32)
    X_valid[col] = X_valid[col].astype(np.int32)

print("Train shape:", X_train.shape)
print("Validation shape:", X_valid.shape)

Train shape: (5747365, 33)
Validation shape: (85372, 33)
